# Mathematische OSS in der Praxis
## Logik, Optimierung, Graphentheorie und mehr mit SageMath

**Manfred Scheucher**  
Grazer Linuxtage 2026

---
## SageMath 

* Freie Software zur Lösung mathematischer Probleme
* vereint Stärken vieler Open-Source-Libraries 
* einheitliches Python-Interface
* [https://doc.sagemath.org/html/en/installation/](https://doc.sagemath.org/html/en/installation/)

## Verschiedene Bereiche 

* Analysis (Kurvendiskussionen)
* Algebra (Gleichungssysteme)
* Geometrie 2D/3D (Visualisierung)
* Graphentheorie (kürzeste Wege, Netzwerkflüsse)
* Optimierungsprobleme
* Logik 
* ...


## Was Erwartet mich in diesem Vortrag?

* schöne Bilder
* viele Beispiele 
* Voraussetzung: minimale Programmier Kenntnisse (Python)

---
## Basics

Wie Verwenden?
* Jupyter Notebook
* CommandLineInterface `sage`
* Script mit Parameter `sage script.sage`

In [1]:
print(f"Hallo Welt!") # Python Input/Output

Hallo Welt!


In [2]:
def f(x): return x*x
print([f(x) for x in range(10)]) # Funktionen, List Comprehension, etc.

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [3]:
%%time 
# Zeit messen in Jupyter/Sage
def fib(x): return x if x<=1 else fib(x-1)+fib(x-2)
print([fib(x) for x in range(32)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418, 317811, 514229, 832040, 1346269]
CPU times: user 1.93 s, sys: 15.1 ms, total: 1.94 s
Wall time: 1.94 s


In [4]:
%%time
# Dynamische Programmierung: Zwischenergebnisse speichern
@cached_function 
def fib(x): return x if x<=1 else fib(x-1)+fib(x-2)
print([fib(x) for x in range(32)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025, 121393, 196418, 317811, 514229, 832040, 1346269]
CPU times: user 107 μs, sys: 8 μs, total: 115 μs
Wall time: 115 μs


* Ausführung auch via commandline: `sage script.sage` bzw `time sage script.sage`
* Parameter `argv` bzw `argparse`

In [5]:
%%time 
import time
def f(i): 
    time.sleep(float(0.1)) 
    return i
print([f(i) for i in range(10)]) # 1.0 Sekunden auf 1 CPU

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
CPU times: user 1.93 ms, sys: 753 μs, total: 2.69 ms
Wall time: 1.04 s


In [6]:
%%time 
@parallel
def f(i): 
    time.sleep(float(0.1)) 
    return i
print(list(f(range(10)))) # 0.1 Sekunden auf 10 CPUs (parallelisiert auf Jupyter)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
CPU times: user 1.36 ms, sys: 3.55 ms, total: 4.91 ms
Wall time: 104 ms


Sideremark: in diesem Beispiel braucht selbst `print(list(f(range(1000))))` nur 0.1 sec da kein busy wait!

In [7]:
from multiprocessing import Pool
def f(i): # muss unterhalb von "from multiprocessing import Pool" definiert werden
    time.sleep(1)
    return i
#print(list(Pool().map(f,range(10)))) # Variante für Sage-Script (sage script.sage), nicht in Jupyter

In [14]:
%%time
!cat pool_example.sage && sage pool_example.sage

from multiprocessing import Pool
def f(i): # muss unterhalb von "from multiprocessing import Pool" definiert werden
    time.sleep(1)
    return iCPU times: user 6.84 ms, sys: 12 ms, total: 18.9 ms
Wall time: 1.19 s


---
## Beispiel 1: Kurvendiskussion


Klassische Matura-Aufgaben

In [9]:
x = var('x')
f = x*sin(x*x)
print(f) # Textuelle Ausgabe
f.show() # Formatierte Ausgabe

x*sin(x^2)


x*sin(x^2)

In [10]:
Nullstellen, Extremwerte, Polynom

NameError: name 'Nullstellen' is not defined

In [ ]:
print("f'(x) =", f.diff()) # Symbolische Ableitung
print("F(x)  =", f.integrate(x)) # Stammfunktion

In [ ]:
# Plot: Fläche unter Kurve
p  = plot(f, (x, 0, pi))
p += plot(f, (x, pi/4, pi/2), fill=True, fillcolor="blue", fillalpha=0.25)
p.show()

In [ ]:
# Flächenbestimmung
I = integrate(f, x, pi/4, pi/2)
print(f"Fläche: {I} ≈ {float(I)}") # Rechnung symbolisch, Ergebnis numerisch umwandeln

In [ ]:
# beliebige Präzision (Anzahl Bits)
for prec in [30,100,300,1000]:
    R = RealField(prec)
    print(f"{prec} -> {R(I)}")

In [ ]:
# Plot: Fläche zwischen Kurven 
f = x*sin(x*x)
g = -sin(x)

p  = plot(f, (x, 0, pi), color='blue')
p += plot(f, (x, pi/4, pi/2), color='blue', fill=True, fillcolor='blue', fillalpha=0.25)

p += plot(g, (x, 0, pi), color='red')
p += plot(g, (x, pi/4, pi/2), color='red', fill=True, fillcolor='red', fillalpha=0.25)
p.show()

In [ ]:
# Symbolische Flächenbestimmung
I = integrate(f-g, x, pi/4, pi/2)
print(f"Fläche: {I} ≈ {float(I)}")

In [ ]:
print((f-g).find_root(1,2)) # Numerische Suche nach Nullstelle in Interval, nur bedingt symbolisch möglich
print((f-g).find_root(2,3))

---
## Gleichungssysteme

Schnitt von zwei Kreisen:

$$C_1: (x-1)^2 + y^2 = 4 \qquad C_2: (x+1)^2 + y^2 = 4$$

In [ ]:
x, y = var('x y')
C1 = (x-1)^2 + y^2 == 4
C2 = (x+1)^2 + y^2 == 4

In [ ]:
solutions = solve([C1, C2], x, y) # Symbolische Schnittpunkte
for s in solutions:
    sx = s[0].rhs()
    sy = s[1].rhs()
    print(f"{s} -> ({sx},{sy}) ≈ ({RR(sx)},{RR(sy)})")

In [ ]:
p  = implicit_plot(C1.lhs() - C1.rhs(), (x, -4, 4), (y, -3, 3), color='blue')
p += implicit_plot(C2.lhs() - C2.rhs(), (x, -4, 4), (y, -3, 3), color='red')
for s in solutions:
    sx = s[0].rhs()
    sy = s[1].rhs()
    p += point((RR(sx), RR(sy)), size=50, color='black', zorder=2)
p.show(aspect_ratio=1)

**Idee hinter symbolischen Lösung:**

In [ ]:
# jeder Schnittpunkt (x,y) ist Nullstelle beider Gleichungen 
f1 = (x-1)^2 + y^2 - 4 
f2 = (x+1)^2 + y^2 - 4 

In [ ]:
# ... und Nullestelle von f3 = f1-f2
f3 = (f1-f2)
print(f3) 
print(f3.simplify_full()) # 4x ist nur Null wenn x=0 

In [ ]:
# ... und Nullstelle von f4 = f1+f2, insbesondere wenn x=0 gesetzt
f4 = (f1+f2)
print(f4)
print(f4(x=0))
print(f4(x=0).roots())

Anm: Lösungen zu finden ist im Allgemeinen schwer, doubly exponential

---
## Geometrie

2D und 3D Strukturen mit wenig Code visualisieren

**Beispiel:** 

In [ ]:
n = 100
point2d([(cos(x*2*pi/n),sin(x*2*pi/n)) for x in range(n)]).show(aspect_ratio=1)

In [ ]:
p = point3d([(cos(x*2*pi/n),sin(x*2*pi/n),0) for x in range(n)],color='blue')
p += point3d([(cos(x*2*pi/n),0,sin(x*2*pi/n)) for x in range(n)],color='red')
p += point3d([(0,cos(x*2*pi/n),sin(x*2*pi/n)) for x in range(n)],color='green')
p.show(aspect_ratio=1)

In [ ]:
import random
from scipy.spatial import Voronoi, ConvexHull

def new_points():
    return [(random.uniform(0,1), random.uniform(0,1)) for _ in range(20)]

pts = new_points()
hull2d = ConvexHull(pts)
hull_indices = set(hull2d.vertices)

@interact
def geo_demo(mode=selector(['Nearest Neighbor', 'Voronoi', 'Convex Hull'], label='Modus'),
             neu=button('Neu würfeln')):
    global pts, hull2d, hull_indices
    if neu:
        pts = new_points()
        hull2d = ConvexHull(pts)
        hull_indices = set(hull2d.vertices)

    inner = [pts[i] for i in range(len(pts)) if i not in hull_indices]
    outer = [pts[i] for i in hull_indices]

    p = points(inner, size=30, color='black', zorder=5)
    p += points(outer, size=30, color='blue', zorder=5)

    if mode == 'Nearest Neighbor':
        for i, (ax, ay) in enumerate(pts):
            best = min((j for j in range(len(pts)) if j != i),
                       key=lambda j: (pts[j][0]-ax)**2 + (pts[j][1]-ay)**2)
            bx, by = pts[best]
            p += arrow((ax, ay), (bx, by), color='blue', width=1, arrowsize=2)

    elif mode == 'Voronoi':
        vor = Voronoi(pts)
        for simplex in vor.ridge_vertices:
            if -1 not in simplex:
                x0, y0 = vor.vertices[simplex[0]]
                x1, y1 = vor.vertices[simplex[1]]
                p += line([(x0,y0),(x1,y1)], color='red')

    elif mode == 'Convex Hull':
        hull_pts = [pts[i] for i in hull2d.vertices]
        p += polygon(hull_pts, color='lightblue', alpha=0.5)
        p += line(hull_pts+[hull_pts[0]], color='blue', thickness=2)

    p.show(axes=False, frame=True, xmin=-0.05, xmax=1.05, ymin=-0.05, ymax=1.05)

In [ ]:
# 3D: Ikosaeder — 20 Flächen, 12 Ecken, wenig Code
I = polytopes.icosahedron()
print("Ecken:", len(I.vertices()))
print("Flächen:", len(I.faces(2)))
I.plot()

In [ ]:
import random
from scipy.spatial import ConvexHull
import numpy as np

random.seed()  # system time
pts3 = [(random.uniform(-1,1), random.uniform(-1,1), random.uniform(-1,1)) for _ in range(30)]
arr3 = np.array(pts3)

hull3 = ConvexHull(arr3)
hull_idx3 = set(hull3.vertices)

p3 = point3d([pts3[i] for i in hull_idx3], size=10, color='blue')
p3 += point3d([pts3[i] for i in range(len(pts3)) if i not in hull_idx3], size=10, color='black')

for simplex in hull3.simplices:
    tri = [pts3[i] for i in simplex]
    p3 += polygon3d(tri, color='lightblue', alpha=0.3)

p3.show()

### 3D Pseudohyperebenen-Arrangement

- [helenabergold.github.io/supp/3d_signotopes](https://helenabergold.github.io/supp/3d_signotopes/index.html)
- [arxiv.org/abs/2303.04079](https://arxiv.org/abs/2303.04079) — Signotopes

### 2D Pseudokreis-Arrangement

[IPE Drawing Editor](https://ipe.otfried.org/)

![arrangement](figs/arrangement.png)

![krupp](figs/Krupp.png)

Flächen → Knoten, Nachbarschaft → Kanten

- [arxiv.org/abs/1708.06449](https://arxiv.org/abs/1708.06449) — Triangles and Drawings
- [arxiv.org/abs/1712.02149](https://arxiv.org/abs/1712.02149) — Circularizability

In [ ]:
from IPython.display import display, HTML

with open('lasagne3d/example_pshyperplanes.html', 'r') as f:
    html_content = f.read()

display(HTML(f'<div style="width:900px; height:600px; overflow:hidden">{html_content}</div>'))

---
## Graphentheorie

Sage hat eine eigene native Graphbibliothek (Cython/C-Backend).  
Optional: Schnittstellen zu **Boost Graph Library** (C++) und **NetworkX** (Python).

```python
G.networkx_graph()   # Konvertierung zu NetworkX
# Boost: sage.graphs.boost_graph (Cython-Wrapper, z.B. für Dijkstra)
```

In [ ]:
# Graph definieren und plotten
G = Graph({'A': ['B','C','D'], 'B': ['C','E'], 'C': ['F'], 'D': ['E'], 'E': ['F'], 'F': []})
print("Knoten:", G.vertices())
print("Kanten:", G.edges(labels=False))
G.plot()

In [ ]:
# Graphfärbung
chi = G.chromatic_number()
print("Chromatische Zahl:", chi)

coloring = G.coloring()
print("Färbung:", coloring)

G.plot(vertex_colors={
    'red':   coloring[0],
    'blue':  coloring[1],
    'green': coloring[2] if len(coloring) > 2 else []
})

In [ ]:
# Kürzeste Wege — Floyd-Warshall (paarweise Distanzmatrix)
D = G.distance_all_pairs()
vertices = sorted(G.vertices())

print("Distanzmatrix (Floyd-Warshall):")
header = "    " + "  ".join(f"{v:>3}" for v in vertices)
print(header)
for u in vertices:
    row = f"{u:>3} " + "  ".join(f"{D[u][v]:>3}" for v in vertices)
    print(row)

### Graphbibliotheken — Ökosystem

| Bibliothek | Sprache | Stärken |
|---|---|---|
| **SageMath** | Python/Cython | CAS-Integration, exakte Algorithmen |
| **NetworkX** | Python | Einfach, viele Algorithmen, gute Docs |
| **Boost Graph Library** | C++ | Performance, Teil von Boost |

SageMath kann zu NetworkX konvertieren und nutzt Boost intern optional als Backend.

---
## Mixed Integer Linear Programming (MILP)

**"Programmierung"** hier = abstraktes mathematisches Konzept (wie auch "dynamische Programmierung") — keine Programmiersprache.

### Modell: Gewinnmaximierung im Stall

Variablen: $x_1$ = Schweine, $x_2$ = Hühner, $x_3$ = Kühe — jeweils **ganzzahlig, nicht-negativ**.

$$\max \; 30 x_1 + 5 x_2 + 50 x_3$$

$$\text{s.t.} \quad
\underbrace{50 x_1 + 10 x_2 + 200 x_3}_{\text{Wasser (L/Tag)}} \leq 1000$$

$$\underbrace{3 x_1 + 0.5 x_2 + 8 x_3}_{\text{Futter (kg/Tag)}} \leq 40$$

$$\underbrace{x_1 + x_2 + x_3}_{\text{Stallplätze}} \leq 12$$

$$x_1, x_2, x_3 \in \mathbb{Z}_{\geq 0}$$

Modellierung oft **straight-forward** — aber: Encoding-Entscheidungen können Solver-Laufzeit drastisch beeinflussen.

In [ ]:
p = MixedIntegerLinearProgram(maximization=True)
x = p.new_variable(integer=True, nonnegative=True)

p.set_objective(30*x[1] + 5*x[2] + 50*x[3])          # max Profit (€)

p.add_constraint(50*x[1] + 10*x[2] + 200*x[3] <= 1000)  # Wasser (L/Tag)
p.add_constraint( 3*x[1] +  0.5*x[2] +  8*x[3] <= 40)   # Futter (kg/Tag)
p.add_constraint(   x[1] +    x[2] +    x[3] <= 12)      # Stallplätze

p.solve()

print(f"Schweine: {int(p.get_values(x[1]))}")
print(f"Hühner:   {int(p.get_values(x[2]))}")
print(f"Kühe:     {int(p.get_values(x[3]))}")
print(f"Profit:   € {int(p.get_values(30*x[1] + 5*x[2] + 50*x[3]))}")

### Encoding & Solver-Effizienz

- Modellierung oft **straight-forward** — aber Encoding-Entscheidungen können Laufzeit **drastisch** beeinflussen
- Wenige Variablen & Constraints = gut; aber: Hilfsvariablen können Suchraum verkleinern
- **Symmetrien** im Suchraum werden von Solvern meist *nicht* erkannt → manuell brechen
- **Dualität**: Maximierungsproblem ↔ äquivalentes Minimierungsproblem
- SAT ist Spezialfall von MILP (nur 0/1-Variablen, Constraints als Summe ≥ 1)

---
## SAT Solver

**Hinweis:** Die SAT-Solver-Implementierungen in Sage sind veraltet (PicoSAT etc.) und nicht mehr zeitgemäß — aktuelle Solver sind CaDiCaL, Kissat, Glucose.

Wir verwenden stattdessen **pysat** — einfach installierbar, gleiche Syntax, aktuelle Solver:

```
pip install python-sat
```

In [ ]:
# pip install python-sat
from pysat.solvers import Cadical195

# Formel: (x1 ∨ x2) ∧ (¬x1 ∨ x3) ∧ (¬x2 ∨ ¬x3)
cnf = [
    [1, 2],    # x1 oder x2
    [-1, 3],   # nicht x1 oder x3
    [-2, -3],  # nicht x2 oder nicht x3
]

solver = Cadical195(bootstrap_with=cnf)
for model in solver.enum_models():
    print(model)

---
## Komplexität & NP

**Häufiges Missverständnis:** NP ≠ "nicht-polynomiell"

NP = **N**ichtdeterministisch **P**olynomiell  
→ bisher kein deterministischer Poly-Algorithmus gefunden, aber P=NP nicht ausgeschlossen!

### Überraschungen der letzten Jahrzehnte

- **Primfaktorisierung**: lange als "quadrathart" eingeschätzt → General Number Field Sieve (GNFS): subexponentiell
- **Graphisomorphie**: lange offen → Babai 2015: quasipolynomiell
- **3SUM**: jahrzehntelang als $\Theta(n^2)$-hart bezeichnet → 2014: $O(n^2 \log\log n / \log n)$

---

Falls die besten Tools nicht durchkommen: manchmal hilft es zu wissen, dass ein Problem **NP-schwer** ist —  
dann MILP/SAT einsetzen statt selbst "Zähne ausbeißen".

### Approximation

Wenn exakt zu teuer: **Approximationsalgorithmen** — für reale Anwendungen oft gut genug.

- **Traveling Salesman (TSP)**: NP-schwer, auch euklidisch — aber $(1+\varepsilon)$-Approximation in Polynomialzeit möglich

### Dynamische Programmierung

Auch "Programmierung" = abstraktes Konzept (keine Sprache).

**Kürzeste Wege** (Rekursion): Weg von $u$ nach $v$ über Nachbarn $w$:
$$d(u,v) = \min_{w \sim v} \bigl( d(u,w) + \text{weight}(w,v) \bigr)$$
Wenn kürzeste Wege zu allen Nachbarn bekannt → optimal kombinieren.

---
## Weiterführende Topics

Was heute nicht dran kam — aber in Sage (und drumherum) verfügbar:

| Topic | Tool/Library | Links |
|---|---|---|
| **SMT Solver** | Z3 (via Python) | https://ericpony.github.io/z3py-tutorial |
| **Zahlentheorie** | Sage nativ | Primzahlen, elliptische Kurven, ... |
| **Gruppentheorie** | Sage / GAP (Backend) | Permutationsgruppen, Charaktertafeln |
| **Formale Verifikation** | Lean, Coq | Beweise maschinell verifizieren |
| **Quantified Boolean Formula** | QCIR, QBF-Solver | https://wikipedia.org/wiki/QBF |
| **Answer Set Programming** | clingo/ASP | https://wikipedia.org/wiki/Answer_set_programming |
| **#SAT / Model Counting** | SharpSAT, ApproxMC | Lösungen zählen statt finden |

---

**Danke!**  
Fragen? Code & Notebook: *(Link folgt)*